# Lake CH4 reconstruction: DCH4 slim Colab notebook

This is the more stable slim version for DCH4. It keeps core predictors but removes the problematic VIIRS `cf_cvg` band that caused Earth Engine homogeneous-image-collection failures.

Core predictors kept: ERA5 climate, SRTM terrain, OpenLandMap soil organic carbon, human modification, GPW population, DMSP nightlights, VIIRS avg_rad nightlights, MODIS land cover, and JRC water occurrence. HydroLAKES, GLOBathy, and water-color proxies stay off by default.


In [ ]:
!pip -q install earthengine-api gdown optuna scikit-learn seaborn

In [ ]:
import json
import time
from pathlib import Path
import warnings

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, train_test_split, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

PROJECT_ID = "wetland-ch4"

# Choose "DCH4" or "ECH4".
DATASET_KIND = "DCH4"

DRIVE_FILE_IDS = {
    "DCH4": "14mDWToCr64FjJzB7iOTTdj3pXk8sIRzF",
    "ECH4": "13NsFNezyf8KD4UlgzIXeDNhxC8lKKWIj",
}

TARGET_COLS = {
    "DCH4": "DCH4_Flux (mmol·m-2·d-1)",
    "ECH4": "ECH4_Flux (mmol·m-2·d-1)",
}

# Set True when changing Earth Engine extraction settings or row limits.
FORCE_REEXTRACT_EE = True

# Use 100 for a quick smoke test, 1000 for a medium run, or None for all valid 2003+ rows.
MAX_ROWS_FOR_EXTRACTION = 1000

# Earth Engine export settings.
EE_EXPORT_FOLDER = "wetland_ch4_gee_exports"
POLL_SECONDS = 60
MAX_WAIT_MINUTES = 240
MISSING_SENTINEL = -9999

# Spatial extraction settings.
POINT_BUFFER_M = 5000
LANDSCAPE_BUFFER_M = 25000
HYDROLAKES_SEARCH_M = 10000

# Feature switches. Keep water color off for the stable first run.
USE_HYDROLAKES = False
USE_GLOBATHY = False
USE_MODIS_OCEAN_COLOR = False
USE_LANDSAT_WATER_COLOR = False
USE_SENTINEL2_WATER_COLOR = False

# Slim stability switch: cf_cvg caused VIIRS mixed integer type failures in GEE.
USE_VIIRS_CF_CVG = False

# Modeling settings.
RANDOM_STATE = 42
TEST_SIZE = 0.2
USE_OPTUNA = False
N_OPTUNA_TRIALS = 40
CV_FOLDS = 5
USE_OBS_WATER_TEMP = False
INCLUDE_COORDINATES = True

if DATASET_KIND not in DRIVE_FILE_IDS:
    raise ValueError("DATASET_KIND must be 'DCH4' or 'ECH4'.")

DRIVE_FILE_ID = DRIVE_FILE_IDS[DATASET_KIND]
TARGET_COL = TARGET_COLS[DATASET_KIND]
FLUX_COL = f"flux_{DATASET_KIND.lower()}"
RUN_TAG = "slim"
RUN_SUFFIX = f"_{RUN_TAG}" if RUN_TAG else ""

from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = Path("/content/drive/MyDrive/wetland_ch4_batch_outputs")
EXPORT_DIR = Path("/content/drive/MyDrive") / EE_EXPORT_FOLDER
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = OUTPUT_DIR / f"{DATASET_KIND}_input_from_drive.csv"
CLEAN_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_clean_2003plus.csv"
ENRICHED_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_batch_enhanced_predictors.csv"
COVERAGE_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_batch_predictor_coverage.csv"
METRICS_JSON = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_paper_style_metrics.json"
TEST_PRED_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_paper_style_test_predictions.csv"
OOB_PRED_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_paper_style_oob_predictions.csv"
FEATURE_IMPORTANCE_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_rf_impurity_importance.csv"
PERMUTATION_IMPORTANCE_CSV = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_rf_permutation_importance.csv"
SCATTER_PNG = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_paper_style_scatter.png"
IMPORTANCE_PNG = OUTPUT_DIR / f"{DATASET_KIND}{RUN_SUFFIX}_paper_style_importance.png"

EXPORT_PREFIX = f"{DATASET_KIND}_gee_enhanced_core{RUN_SUFFIX}"
EE_EXPORT_CSV = EXPORT_DIR / f"{EXPORT_PREFIX}.csv"

print("Dataset:", DATASET_KIND)
print("Target:", TARGET_COL)
print("Output dir:", OUTPUT_DIR)
print("EE export CSV:", EE_EXPORT_CSV)

## 1. Download and clean the table

In [ ]:
if not INPUT_CSV.exists():
    print("Downloading source CSV...")
    gdown.download(f"https://drive.google.com/uc?id={DRIVE_FILE_ID}", str(INPUT_CSV), quiet=False)
else:
    print("Using cached source CSV:", INPUT_CSV)

raw = pd.read_csv(INPUT_CSV)
print("Raw shape:", raw.shape)
display(raw.head())
print(raw.columns.tolist())

In [ ]:
def parse_month_start(series):
    dt = pd.to_datetime(series, errors="coerce")
    return dt.dt.to_period("M").dt.to_timestamp()


df = raw.copy()
df["Lake_name"] = df["Lake_name"].astype(str).str.replace("\u00a0", " ", regex=False).str.strip()
df["date_start"] = parse_month_start(df["Date_start"])
df["date_end_month"] = parse_month_start(df["Date_end"])
df["date_end_exclusive"] = df["date_end_month"] + pd.offsets.MonthBegin(1)
df["date_end_inclusive"] = df["date_end_exclusive"] - pd.Timedelta(days=1)
df["date_mid"] = df["date_start"] + (df["date_end_inclusive"] - df["date_start"]) / 2

df[FLUX_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df["lat"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["lon"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["water_temp_obs_C"] = pd.to_numeric(
    df["Water temperature (℃)"].astype(str).str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce",
)

valid = (
    df["date_start"].notna()
    & df["date_end_exclusive"].notna()
    & (df["date_end_exclusive"] > df["date_start"])
    & df["lat"].between(-90, 90)
    & df["lon"].between(-180, 180)
    & df[FLUX_COL].notna()
    & (df[FLUX_COL] >= 0)
    & (df["date_start"].dt.year >= 2003)
)

df = df.loc[valid].copy().reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=int)
df["date_start_ee"] = df["date_start"].dt.strftime("%Y-%m-%d")
df["date_end_ee"] = df["date_end_exclusive"].dt.strftime("%Y-%m-%d")
df["mid_year_int"] = df["date_mid"].dt.year.astype(int)
df["mid_year"] = df["date_mid"].dt.year + (df["date_mid"].dt.dayofyear - 1) / 365.25
df["mid_month"] = df["date_mid"].dt.month
df["duration_days"] = (df["date_end_exclusive"] - df["date_start"]).dt.days
df["month_sin"] = np.sin(2 * np.pi * df["mid_month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["mid_month"] / 12)

GPW_YEARS = np.array([2000, 2005, 2010, 2015, 2020])
df["pop_year"] = df["mid_year_int"].apply(lambda y: int(GPW_YEARS[np.argmin(np.abs(GPW_YEARS - y))]))
df["mcd12_year"] = df["mid_year_int"].clip(lower=2001, upper=2024)
df["jrc_year"] = df["mid_year_int"].clip(lower=1984, upper=2021)

if MAX_ROWS_FOR_EXTRACTION is not None:
    df = df.head(int(MAX_ROWS_FOR_EXTRACTION)).copy()

df.to_csv(CLEAN_CSV, index=False)
print("Clean rows:", len(df))
print("Date range:", df["date_start"].min(), "to", df["date_end_inclusive"].max())
print("Flux range:", df[FLUX_COL].min(), "to", df[FLUX_COL].max())
display(df[["row_id", "Lake_name", "lat", "lon", "date_start_ee", "date_end_ee", FLUX_COL]].head())

## 2. Initialize Earth Engine

In [ ]:
import ee

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized:", PROJECT_ID)

## 3. Earth Engine datasets and helper functions

In [ ]:
# Time-varying products
ERA5 = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
VIIRS = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG")
DMSP = ee.ImageCollection("NOAA/DMSP-OLS/NIGHTTIME_LIGHTS")
GPW = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density")
MCD12Q1 = ee.ImageCollection("MODIS/061/MCD12Q1")
JRC_YEARLY = ee.ImageCollection("JRC/GSW1_4/YearlyHistory")
MODIS_OC = ee.ImageCollection("NASA/OCEANDATA/MODIS-Aqua/L3SMI")

# Static products
SRTM = ee.Image("CGIAR/SRTM90_V4").select("elevation")
TERRAIN = ee.Image.cat([
    SRTM.rename("srtm_elevation_m"),
    ee.Terrain.slope(SRTM).rename("srtm_slope_deg"),
])
SOC = ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02")
GHM = ee.ImageCollection("CSP/HM/GlobalHumanModification").first().select("gHM").rename("human_modification")

if USE_HYDROLAKES:
    HYDROLAKES = ee.FeatureCollection("projects/sat-io/open-datasets/HydroLakes/lake_poly_v10")
if USE_GLOBATHY:
    GLOBATHY = ee.Image("projects/sat-io/open-datasets/GLOBathy/GLOBathy_bathymetry").select("b1").rename("globathy_max_depth_m")

# Optional water color products
LT05 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
LE07 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
LC08 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
LC09 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

ERA5_TEMP_K_BANDS = [
    "temperature_2m", "dewpoint_temperature_2m", "skin_temperature",
    "lake_mix_layer_temperature", "lake_bottom_temperature",
]
ERA5_OTHER_BANDS = [
    "total_precipitation_sum", "potential_evaporation_sum", "runoff_sum",
    "surface_solar_radiation_downwards_sum", "surface_thermal_radiation_downwards_sum",
    "surface_pressure", "u_component_of_wind_10m", "v_component_of_wind_10m",
    "volumetric_soil_water_layer_1", "lake_mix_layer_depth", "lake_ice_depth",
]
ERA5_BANDS = [b + "_C" for b in ERA5_TEMP_K_BANDS] + ERA5_OTHER_BANDS

MODIS_OC_BANDS = [
    "chlor_a", "Rrs_412", "Rrs_443", "Rrs_469", "Rrs_488", "Rrs_531",
    "Rrs_547", "Rrs_555", "Rrs_645", "Rrs_667", "Rrs_678", "nflh", "poc", "sst",
]
DMSP_BANDS = ["stable_lights", "avg_vis", "cf_cvg", "avg_lights_x_pct"]
VIIRS_BANDS = ["avg_rad"] + (["cf_cvg"] if USE_VIIRS_CF_CVG else [])
HYDRO_ATTRS = [
    "Hylak_id", "Lake_area", "Shore_len", "Shore_dev", "Vol_total",
    "Depth_avg", "Dis_avg", "Res_time", "Elevation", "Slope_100",
    "Wshd_area", "Lake_type",
]


def prep_era5(img):
    temp_c = img.select(ERA5_TEMP_K_BANDS).subtract(273.15).rename([b + "_C" for b in ERA5_TEMP_K_BANDS])
    return temp_c.addBands(img.select(ERA5_OTHER_BANDS)).copyProperties(img, ["system:time_start"])


ERA5P = ERA5.map(prep_era5)
MCD12_MODE = MCD12Q1.select("LC_Type1").mode().rename("LC_Type1")
JRC_MODE = JRC_YEARLY.select("waterClass").mode().rename("waterClass")


def clean_value(v):
    return ee.Algorithms.If(ee.Algorithms.IsEqual(v, None), MISSING_SENTINEL, v)


def prefix_dictionary(d, prefix):
    d = ee.Dictionary(d)
    keys = d.keys()
    values = keys.map(lambda k: clean_value(d.get(k)))
    new_keys = keys.map(lambda k: ee.String(prefix).cat(ee.String(k)))
    return ee.Dictionary.fromLists(new_keys, values)


def masked_image(bands):
    return ee.Image.constant(ee.List.repeat(0, len(bands))).rename(bands).updateMask(ee.Image.constant(0))


def cast_image_to_float(img):
    return ee.Image(img).toFloat().copyProperties(img, ["system:time_start"])


def collection_mean_or_masked(ic, bands):
    # Some products, especially VIIRS cf_cvg, change integer type across years.
    # Casting avoids Earth Engine homogeneous-collection export failures.
    sub = ic.select(bands).map(cast_image_to_float)
    return ee.Image(ee.Algorithms.If(sub.size().gt(0), sub.mean(), masked_image(bands).toFloat()))


def reduce_image_mean_dict(img, bands, geom, scale, prefix):
    reducer = ee.Reducer.mean().combine(reducer2=ee.Reducer.count(), sharedInputs=True)
    stats = ee.Dictionary(img.select(bands).reduceRegion(
        reducer=reducer,
        geometry=geom,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
        tileScale=4,
    ))
    keys = ee.List(bands)
    out_keys = keys.map(lambda b: ee.String(prefix).cat(ee.String(b)))
    out_vals = keys.map(
        lambda b: ee.Algorithms.If(
            ee.Number(stats.get(ee.String(b).cat("_count"), 0)).gt(0),
            clean_value(stats.get(ee.String(b).cat("_mean"))),
            MISSING_SENTINEL,
        )
    )
    return ee.Dictionary.fromLists(out_keys, out_vals)


def temporal_fallback_mean(ic, bands, start, end, geom, scale, prefix, long_start, long_end):
    # target window -> +/- one year -> multi-year mean
    target_ic = ic.filterDate(start, end)
    near_ic = ic.filterDate(start.advance(-1, "year"), end.advance(1, "year"))
    long_ic = ic.filterDate(long_start, long_end)
    img = (
        collection_mean_or_masked(target_ic, bands)
        .unmask(collection_mean_or_masked(near_ic, bands))
        .unmask(collection_mean_or_masked(long_ic, bands))
    )
    d = reduce_image_mean_dict(img, bands, geom, scale, prefix)
    counts = ee.Dictionary({
        prefix + "target_image_count": target_ic.size(),
        prefix + "near_image_count": near_ic.size(),
        prefix + "long_image_count": long_ic.size(),
    })
    return d.combine(counts, True)

## 4. Core predictor reducers

In [ ]:
def dict_from_feature_or_missing(feat, keys):
    keys = ee.List(keys)
    d = ee.Dictionary(feat.toDictionary(keys))
    vals = keys.map(lambda k: clean_value(d.get(k)))
    return ee.Dictionary.fromLists(keys, vals)


def reduce_hydrolakes(point_geom):
    keys = HYDRO_ATTRS + ["distance_m"]
    missing = ee.Dictionary.fromLists(keys, ee.List.repeat(MISSING_SENTINEL, len(keys)))
    candidates = HYDROLAKES.filterBounds(point_geom.buffer(HYDROLAKES_SEARCH_M))

    def add_distance(feat):
        return feat.set("distance_m", feat.geometry().distance(point_geom, 1))

    nearest = ee.Feature(candidates.map(add_distance).sort("distance_m").first())
    nearest_dict = dict_from_feature_or_missing(nearest, keys)
    d = ee.Dictionary(ee.Algorithms.If(candidates.size().gt(0), nearest_dict, missing))
    return prefix_dictionary(d, "hydrolakes_")


def reduce_static(point_geom, point_buffer, landscape_buffer):
    soc_scaled = SOC.select(["b0", "b10", "b30", "b60", "b100", "b200"]).multiply(5).rename([
        "soc_0cm_gkg", "soc_10cm_gkg", "soc_30cm_gkg",
        "soc_60cm_gkg", "soc_100cm_gkg", "soc_200cm_gkg",
    ])
    d = ee.Dictionary({})
    if USE_HYDROLAKES:
        d = d.combine(reduce_hydrolakes(point_geom), True)
    d = d.combine(reduce_image_mean_dict(TERRAIN, ["srtm_elevation_m", "srtm_slope_deg"], point_buffer, 90, "terrain_"), True)
    d = d.combine(reduce_image_mean_dict(soc_scaled, [
        "soc_0cm_gkg", "soc_10cm_gkg", "soc_30cm_gkg",
        "soc_60cm_gkg", "soc_100cm_gkg", "soc_200cm_gkg",
    ], landscape_buffer, 250, "soil_"), True)
    d = d.combine(reduce_image_mean_dict(GHM, ["human_modification"], landscape_buffer, 1000, "ghm_"), True)
    if USE_GLOBATHY:
        d = d.combine(reduce_image_mean_dict(GLOBATHY, ["globathy_max_depth_m"], point_buffer, 30, "globathy_"), True)
    return d


def reduce_gpw(pop_year, geom):
    ic = GPW.filter(ee.Filter.calendarRange(pop_year, pop_year, "year")).select(["population_density"])
    img = collection_mean_or_masked(ic, ["population_density"])
    return reduce_image_mean_dict(img, ["population_density"], geom, 927, "gpw_").combine(
        ee.Dictionary({"gpw_year_used": pop_year, "gpw_target_image_count": ic.size()}), True
    )


def reduce_dmsp(start_year, end_year, geom):
    target_ic = DMSP.filter(ee.Filter.calendarRange(start_year, end_year, "year")).select(DMSP_BANDS)
    near_ic = DMSP.filter(ee.Filter.calendarRange(start_year.subtract(1), end_year.add(1), "year")).select(DMSP_BANDS)
    long_ic = DMSP.select(DMSP_BANDS)
    img = (
        collection_mean_or_masked(target_ic, DMSP_BANDS)
        .unmask(collection_mean_or_masked(near_ic, DMSP_BANDS))
        .unmask(collection_mean_or_masked(long_ic, DMSP_BANDS))
    )
    return reduce_image_mean_dict(img, DMSP_BANDS, geom, 927, "dmsp_").combine(
        ee.Dictionary({
            "dmsp_target_image_count": target_ic.size(),
            "dmsp_near_image_count": near_ic.size(),
            "dmsp_long_image_count": long_ic.size(),
        }),
        True,
    )


def reduce_mcd12(year, geom):
    sub = MCD12Q1.filter(ee.Filter.calendarRange(year, year, "year")).select("LC_Type1")
    lc = ee.Image(ee.Algorithms.If(sub.size().gt(0), sub.first(), MCD12_MODE)).select("LC_Type1")
    masks = ee.Image.cat([
        lc.gte(1).And(lc.lte(5)).rename("forest_frac"),
        lc.eq(12).Or(lc.eq(14)).rename("cropland_frac"),
        lc.eq(10).rename("grassland_frac"),
        lc.eq(11).rename("wetland_frac"),
        lc.eq(13).rename("urban_frac"),
        lc.eq(17).rename("water_frac"),
        lc.eq(16).rename("barren_frac"),
    ])
    return reduce_image_mean_dict(masks, [
        "forest_frac", "cropland_frac", "grassland_frac", "wetland_frac",
        "urban_frac", "water_frac", "barren_frac",
    ], geom, 500, "mcd12_").combine(
        ee.Dictionary({"mcd12_year_used": year, "mcd12_target_image_count": sub.size()}), True
    )


def reduce_jrc(year, geom):
    sub = JRC_YEARLY.filter(ee.Filter.calendarRange(year, year, "year")).select("waterClass")
    wc = ee.Image(ee.Algorithms.If(sub.size().gt(0), sub.first(), JRC_MODE)).select("waterClass")
    masks = ee.Image.cat([
        wc.eq(2).rename("seasonal_water_frac"),
        wc.eq(3).rename("permanent_water_frac"),
        wc.eq(2).Or(wc.eq(3)).rename("any_water_frac"),
    ])
    return reduce_image_mean_dict(masks, ["seasonal_water_frac", "permanent_water_frac", "any_water_frac"], geom, 30, "jrc_").combine(
        ee.Dictionary({"jrc_year_used": year, "jrc_target_image_count": sub.size()}), True
    )

## 5. Optional water-color proxy reducers, disabled by default

In [ ]:
def mask_landsat_l2(img):
    qa = img.select("QA_PIXEL")
    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )
    return img.updateMask(mask)


def prep_landsat_457(img):
    img = mask_landsat_l2(img)
    return img.select(
        ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7"],
        ["blue", "green", "red", "nir", "swir1", "swir2"],
    ).multiply(0.0000275).add(-0.2).copyProperties(img, ["system:time_start"])


def prep_landsat_89(img):
    img = mask_landsat_l2(img)
    return img.select(
        ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"],
        ["blue", "green", "red", "nir", "swir1", "swir2"],
    ).multiply(0.0000275).add(-0.2).copyProperties(img, ["system:time_start"])


def add_landsat_indices(img):
    ndwi = img.normalizedDifference(["green", "nir"]).rename("ndwi")
    mndwi = img.normalizedDifference(["green", "swir1"]).rename("mndwi")
    ndvi = img.normalizedDifference(["nir", "red"]).rename("ndvi")
    ndti = img.normalizedDifference(["red", "green"]).rename("ndti")
    rg = img.expression("red/(green+1e-6)", {"red": img.select("red"), "green": img.select("green")}).rename("red_green_ratio")
    bg = img.expression("blue/(green+1e-6)", {"blue": img.select("blue"), "green": img.select("green")}).rename("blue_green_ratio")
    nr = img.expression("nir/(red+1e-6)", {"nir": img.select("nir"), "red": img.select("red")}).rename("nir_red_ratio")
    water = ndwi.gt(0).Or(mndwi.gt(0))
    return img.addBands([ndwi, mndwi, ndvi, ndti, rg, bg, nr]).updateMask(water)


LANDSAT_WATER_BANDS = [
    "blue", "green", "red", "nir", "swir1", "swir2",
    "ndwi", "mndwi", "ndvi", "ndti", "red_green_ratio", "blue_green_ratio", "nir_red_ratio",
]


def landsat_collection(start, end, geom):
    return (
        LT05.filterDate(start, end).filterBounds(geom).map(prep_landsat_457)
        .merge(LE07.filterDate(start, end).filterBounds(geom).map(prep_landsat_457))
        .merge(LC08.filterDate(start, end).filterBounds(geom).map(prep_landsat_89))
        .merge(LC09.filterDate(start, end).filterBounds(geom).map(prep_landsat_89))
        .map(add_landsat_indices)
    )


def reduce_landsat(start, end, geom):
    target_ic = landsat_collection(start, end, geom)
    near_ic = landsat_collection(start.advance(-1, "year"), end.advance(1, "year"), geom)
    long_ic = landsat_collection("2003-01-01", "2025-01-01", geom)
    img = (
        collection_mean_or_masked(target_ic, LANDSAT_WATER_BANDS)
        .unmask(collection_mean_or_masked(near_ic, LANDSAT_WATER_BANDS))
        .unmask(collection_mean_or_masked(long_ic, LANDSAT_WATER_BANDS))
    )
    return reduce_image_mean_dict(img, LANDSAT_WATER_BANDS, geom, 30, "landsat_water_").combine(
        ee.Dictionary({
            "landsat_water_target_image_count": target_ic.size(),
            "landsat_water_near_image_count": near_ic.size(),
            "landsat_water_long_image_count": long_ic.size(),
        }),
        True,
    )

## 6. Build FeatureCollection and start batch export

In [ ]:
def enrich_feature(feature):
    f = ee.Feature(feature)
    point = f.geometry()
    point_buffer = point.buffer(POINT_BUFFER_M)
    landscape_buffer = point.buffer(LANDSCAPE_BUFFER_M)

    start = ee.Date(f.get("date_start_ee"))
    end = ee.Date(f.get("date_end_ee"))
    start_year = ee.Number(f.get("year_start")).int()
    end_year = ee.Number(f.get("year_end")).int()
    pop_year = ee.Number(f.get("pop_year")).int()
    mcd12_year = ee.Number(f.get("mcd12_year")).int()
    jrc_year = ee.Number(f.get("jrc_year")).int()

    out = ee.Dictionary({})
    out = out.combine(reduce_static(point, point_buffer, landscape_buffer), True)
    out = out.combine(temporal_fallback_mean(ERA5P, ERA5_BANDS, start, end, point_buffer, 11132, "era5_", "2003-01-01", "2025-01-01"), True)
    out = out.combine(temporal_fallback_mean(VIIRS, VIIRS_BANDS, start, end, landscape_buffer, 464, "viirs_", "2014-01-01", "2025-01-01"), True)
    out = out.combine(reduce_dmsp(start_year, end_year, landscape_buffer), True)
    out = out.combine(reduce_gpw(pop_year, landscape_buffer), True)
    out = out.combine(reduce_mcd12(mcd12_year, landscape_buffer), True)
    out = out.combine(reduce_jrc(jrc_year, point_buffer), True)

    if USE_MODIS_OCEAN_COLOR:
        out = out.combine(temporal_fallback_mean(MODIS_OC, MODIS_OC_BANDS, start, end, point_buffer, 4616, "modis_oc_", "2003-01-01", "2022-03-01"), True)
    if USE_LANDSAT_WATER_COLOR:
        out = out.combine(reduce_landsat(start, end, point_buffer), True)

    return f.set(out)


def df_to_feature_collection(input_df):
    features = []
    for _, r in input_df.iterrows():
        props = {
            "row_id": int(r["row_id"]),
            "date_start_ee": str(r["date_start_ee"]),
            "date_end_ee": str(r["date_end_ee"]),
            "year_start": int(r["date_start"].year),
            "year_end": int((r["date_end_exclusive"] - pd.Timedelta(days=1)).year),
            "pop_year": int(r["pop_year"]),
            "mcd12_year": int(r["mcd12_year"]),
            "jrc_year": int(r["jrc_year"]),
        }
        geom = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
        features.append(ee.Feature(geom, props))
    return ee.FeatureCollection(features)


def wait_for_task(task, poll_seconds=60, max_wait_minutes=240):
    start_time = time.time()
    while True:
        status = task.status()
        state = status.get("state")
        print("Task state:", state)
        if state in ["COMPLETED", "FAILED", "CANCELLED"]:
            print(json.dumps(status, indent=2))
            if state != "COMPLETED":
                msg = status.get("error_message", "")
                raise RuntimeError(f"Earth Engine export did not complete: {state}. {msg}")
            return
        if (time.time() - start_time) / 60 > max_wait_minutes:
            raise TimeoutError("Timed out waiting for Earth Engine export.")
        time.sleep(poll_seconds)

In [ ]:
if FORCE_REEXTRACT_EE:
    for old_file in [ENRICHED_CSV, EE_EXPORT_CSV]:
        if old_file.exists():
            old_file.unlink()
            print("Removed stale file:", old_file)

if ENRICHED_CSV.exists() and not FORCE_REEXTRACT_EE:
    print("Using cached enriched CSV:", ENRICHED_CSV)
else:
    if EE_EXPORT_CSV.exists() and not FORCE_REEXTRACT_EE:
        print("Using existing Earth Engine export CSV:", EE_EXPORT_CSV)
    else:
        print("Starting Earth Engine batch export. This may take a while.")
        fc = df_to_feature_collection(df)
        enriched_fc = fc.map(enrich_feature)
        task = ee.batch.Export.table.toDrive(
            collection=enriched_fc,
            description=f"{EXPORT_PREFIX}_{int(time.time())}",
            folder=EE_EXPORT_FOLDER,
            fileNamePrefix=EXPORT_PREFIX,
            fileFormat="CSV",
        )
        task.start()
        wait_for_task(task, poll_seconds=POLL_SECONDS, max_wait_minutes=MAX_WAIT_MINUTES)

        # Drive mount sometimes needs a moment to show the exported file.
        for _ in range(20):
            if EE_EXPORT_CSV.exists():
                break
            print("Waiting for exported CSV to appear in mounted Drive...")
            time.sleep(15)

    if not EE_EXPORT_CSV.exists():
        raise FileNotFoundError(f"Expected export not found: {EE_EXPORT_CSV}")

    ee_features = pd.read_csv(EE_EXPORT_CSV)
    if ".geo" in ee_features.columns:
        ee_features = ee_features.drop(columns=[".geo"])

    # Earth Engine can export row_id as string/float; normalize before merge.
    ee_features["row_id"] = pd.to_numeric(ee_features["row_id"], errors="coerce").astype("Int64")

    # Drop original metadata columns exported by EE so pandas does not create _x/_y date columns.
    duplicate_cols = [c for c in ee_features.columns if c in df.columns and c != "row_id"]
    if duplicate_cols:
        ee_features = ee_features.drop(columns=duplicate_cols)
        print("Dropped duplicate EE metadata columns:", duplicate_cols)

    enriched = df.merge(ee_features, on="row_id", how="left")

    predictor_prefixes = (
        "hydrolakes_", "terrain_", "soil_", "ghm_", "globathy_",
        "era5_", "viirs_", "dmsp_", "gpw_", "mcd12_", "jrc_",
        "modis_oc_", "landsat_water_", "s2_water_",
    )
    predictor_cols = [c for c in enriched.columns if c.startswith(predictor_prefixes)]
    for c in predictor_cols:
        enriched[c] = pd.to_numeric(enriched[c], errors="coerce")
        enriched.loc[enriched[c] <= MISSING_SENTINEL, c] = np.nan

    enriched.to_csv(ENRICHED_CSV, index=False)

enriched = pd.read_csv(ENRICHED_CSV)
print("Enriched shape:", enriched.shape)
display(enriched.head())

## 7. Coverage and feature selection

In [ ]:
print("ENRICHED_CSV exists:", ENRICHED_CSV.exists(), ENRICHED_CSV)
print("EE_EXPORT_CSV exists:", EE_EXPORT_CSV.exists(), EE_EXPORT_CSV)

if ENRICHED_CSV.exists():
    enriched = pd.read_csv(ENRICHED_CSV)
    print("Loaded enriched:", enriched.shape)
else:
    raise RuntimeError("ENRICHED_CSV does not exist. Run section 6 successfully before section 7.")
predictor_prefixes = (
    "hydrolakes_", "terrain_", "soil_", "ghm_", "globathy_",
    "era5_", "viirs_", "dmsp_", "gpw_", "mcd12_", "jrc_",
    "modis_oc_", "landsat_water_", "s2_water_",
)

predictor_cols = [c for c in enriched.columns if c.startswith(predictor_prefixes)]
if len(predictor_cols) == 0:
    raise ValueError("No predictor columns found. The Earth Engine export/merge did not work.")

for c in predictor_cols:
    enriched[c] = pd.to_numeric(enriched[c], errors="coerce")
    enriched.loc[enriched[c] <= MISSING_SENTINEL, c] = np.nan

coverage = (
    enriched[predictor_cols]
    .notna()
    .mean()
    .sort_values(ascending=False)
    .rename("non_missing_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)
coverage.to_csv(COVERAGE_CSV, index=False)
display(coverage)

coverage_group_summary = (
    coverage.assign(group=coverage["feature"].str.extract(r"^([^_]+)_", expand=False))
    .groupby("group")["non_missing_fraction"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean", ascending=False)
)
print("Coverage by predictor group:")
display(coverage_group_summary)

MIN_FEATURE_COVERAGE = 0.05
kept_remote_cols = coverage.loc[coverage["non_missing_fraction"] >= MIN_FEATURE_COVERAGE, "feature"].tolist()

drop_exact = {"hydrolakes_Hylak_id"}
drop_contains = ("_image_count", "_year_used")
kept_remote_cols = [
    c for c in kept_remote_cols
    if c not in drop_exact and not any(x in c for x in drop_contains)
]

print("Predictors before coverage filter:", len(predictor_cols))
print("Predictors after coverage filter:", len(kept_remote_cols))
print(kept_remote_cols[:40])

## 8. Paper-style train/test RF model

In [ ]:
time_feature_cols = ["mid_year", "mid_month", "duration_days", "month_sin", "month_cos"]
base_feature_cols = time_feature_cols.copy()
if INCLUDE_COORDINATES:
    base_feature_cols += ["lat", "lon"]
if USE_OBS_WATER_TEMP:
    base_feature_cols += ["water_temp_obs_C"]

feature_cols = base_feature_cols + kept_remote_cols

model_df = enriched.dropna(subset=[FLUX_COL]).copy()
model_df = model_df[model_df[FLUX_COL] >= 0].copy()
model_df = model_df.loc[model_df[kept_remote_cols].notna().sum(axis=1) >= 5].copy()

X = model_df[feature_cols].copy()
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")

y_raw = model_df[FLUX_COL].astype(float)
y_log = np.log1p(y_raw)

X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw, idx_train, idx_test = train_test_split(
    X, y_log, y_raw, model_df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Model rows:", len(model_df))
print("Features:", len(feature_cols))
print("Train/test:", len(X_train), len(X_test))

In [ ]:
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


def make_pipeline(params):
    rf = RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        bootstrap=True,
        oob_score=True,
        **params,
    )
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", rf),
    ])


if USE_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 400, 1200, step=100),
            "max_depth": trial.suggest_int("max_depth", 6, 40),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.4, 0.6, 0.8, 1.0]),
        }
        pipe = make_pipeline(params)
        cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        return float(np.mean(cross_val_score(pipe, X_train, y_train_log, cv=cv, scoring="r2", n_jobs=1)))

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    best_params = study.best_params
    print("Best CV R2:", study.best_value)
    print("Best params:", best_params)
else:
    best_params = {
        "n_estimators": 1000,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    }

model = make_pipeline(best_params)
model.fit(X_train, y_train_log)
rf = model.named_steps["rf"]
print("Training OOB R2, log scale:", rf.oob_score_)

## 9. Metrics, predictions, OOB, importance

In [ ]:
y_test_pred_log = model.predict(X_test)
y_test_pred_raw = np.expm1(y_test_pred_log)

metrics = {
    "dataset_kind": DATASET_KIND,
    "target": TARGET_COL,
    "n_total_model_rows": int(len(model_df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(len(feature_cols)),
    "test_size": float(TEST_SIZE),
    "random_state": int(RANDOM_STATE),
    "use_optuna": bool(USE_OPTUNA),
    "n_optuna_trials": int(N_OPTUNA_TRIALS) if USE_OPTUNA else 0,
    "best_params": best_params,
    "rf_oob_r2_train_log_scale": float(rf.oob_score_),
    "test_r2_log_scale": float(r2_score(y_test_log, y_test_pred_log)),
    "test_rmse_log_scale": rmse(y_test_log, y_test_pred_log),
    "test_mae_log_scale": float(mean_absolute_error(y_test_log, y_test_pred_log)),
    "test_r2_original_scale": float(r2_score(y_test_raw, y_test_pred_raw)),
    "test_rmse_original_scale": rmse(y_test_raw, y_test_pred_raw),
    "test_mae_original_scale": float(mean_absolute_error(y_test_raw, y_test_pred_raw)),
}

print(json.dumps(metrics, indent=2))
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)


def existing_or_fallback(df_obj, preferred, fallbacks):
    if preferred in df_obj.columns:
        return preferred
    for c in fallbacks:
        if c in df_obj.columns:
            return c
    raise KeyError(f"Missing expected column {preferred}; tried {fallbacks}")

start_col = existing_or_fallback(model_df, "date_start_ee", ["date_start_ee_x", "date_start_ee_y"])
end_col = existing_or_fallback(model_df, "date_end_ee", ["date_end_ee_x", "date_end_ee_y"])
identity_cols = ["row_id", "Lake_name", "lat", "lon", start_col, end_col, FLUX_COL]
test_pred_df = model_df.loc[idx_test, identity_cols].copy()
test_pred_df = test_pred_df.rename(columns={start_col: "date_start_ee", end_col: "date_end_ee"})
test_pred_df["test_pred_log1p_flux"] = y_test_pred_log
test_pred_df["test_pred_flux"] = y_test_pred_raw
test_pred_df["test_residual_flux"] = test_pred_df[FLUX_COL] - test_pred_df["test_pred_flux"]
test_pred_df.to_csv(TEST_PRED_CSV, index=False)
display(test_pred_df.head())

In [ ]:
X_all_imp = model.named_steps["imputer"].fit_transform(X)
rf_all = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, bootstrap=True, oob_score=True, **best_params)
rf_all.fit(X_all_imp, y_log)
oob_pred_log = rf_all.oob_prediction_
valid_oob = np.isfinite(oob_pred_log)
oob_pred_raw = np.expm1(oob_pred_log)

oob_metrics = {
    "oob_r2_log_scale_all_data": float(r2_score(y_log[valid_oob], oob_pred_log[valid_oob])),
    "oob_rmse_log_scale_all_data": rmse(y_log[valid_oob], oob_pred_log[valid_oob]),
    "oob_r2_original_scale_all_data": float(r2_score(y_raw[valid_oob], oob_pred_raw[valid_oob])),
    "oob_rmse_original_scale_all_data": rmse(y_raw[valid_oob], oob_pred_raw[valid_oob]),
}
metrics.update(oob_metrics)
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(oob_metrics, indent=2))

oob_df = model_df[identity_cols].copy()
oob_df = oob_df.rename(columns={start_col: "date_start_ee", end_col: "date_end_ee"})
oob_df["oob_pred_log1p_flux"] = oob_pred_log
oob_df["oob_pred_flux"] = oob_pred_raw
oob_df["oob_residual_flux"] = oob_df[FLUX_COL] - oob_df["oob_pred_flux"]
oob_df.to_csv(OOB_PRED_CSV, index=False)

importance = pd.DataFrame({"feature": feature_cols, "importance": rf.feature_importances_}).sort_values("importance", ascending=False).reset_index(drop=True)
importance.to_csv(FEATURE_IMPORTANCE_CSV, index=False)
display(importance.head(40))

In [ ]:
perm = permutation_importance(
    model, X_test, y_test_log, scoring="r2",
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1,
)
perm_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
perm_df.to_csv(PERMUTATION_IMPORTANCE_CSV, index=False)
display(perm_df.head(40))

## 10. Plots and saved outputs

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

axes[0].scatter(y_test_log, y_test_pred_log, s=20, alpha=0.6, edgecolor="none")
lims = [min(y_test_log.min(), y_test_pred_log.min()), max(y_test_log.max(), y_test_pred_log.max())]
axes[0].plot(lims, lims, color="black", lw=1)
axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].set_xlabel(f"Observed log1p({DATASET_KIND} flux)")
axes[0].set_ylabel(f"Predicted log1p({DATASET_KIND} flux)")
axes[0].set_title(f"Test log: R2={metrics['test_r2_log_scale']:.3f}, RMSE={metrics['test_rmse_log_scale']:.3f}")

axes[1].scatter(y_test_raw, y_test_pred_raw, s=20, alpha=0.6, edgecolor="none")
raw_lims = [0, np.nanpercentile(np.concatenate([np.asarray(y_test_raw), y_test_pred_raw]), 98)]
axes[1].plot(raw_lims, raw_lims, color="black", lw=1)
axes[1].set_xlim(raw_lims)
axes[1].set_ylim(raw_lims)
axes[1].set_xlabel(f"Observed {DATASET_KIND} flux")
axes[1].set_ylabel(f"Predicted {DATASET_KIND} flux")
axes[1].set_title(f"Original: R2={metrics['test_r2_original_scale']:.3f}, RMSE={metrics['test_rmse_original_scale']:.3f}")
fig.tight_layout()
fig.savefig(SCATTER_PNG, dpi=220)
plt.show()

topn = perm_df.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(topn["feature"], topn["importance_mean"], xerr=topn["importance_std"])
ax.set_xlabel("Permutation importance on test R2")
ax.set_title(f"{DATASET_KIND} top predictors")
fig.tight_layout()
fig.savefig(IMPORTANCE_PNG, dpi=220)
plt.show()

print("Saved files:")
for p in [
    CLEAN_CSV, ENRICHED_CSV, COVERAGE_CSV, METRICS_JSON, TEST_PRED_CSV,
    OOB_PRED_CSV, FEATURE_IMPORTANCE_CSV, PERMUTATION_IMPORTANCE_CSV,
    SCATTER_PNG, IMPORTANCE_PNG,
]:
    print(p)

## Notes

- 和论文对比时，优先看 `test_r2_log_scale` 与 `test_rmse_log_scale`。
- 真实尺度指标用于直观解释，但会被少数极高通量样点强烈影响。
- 默认关闭水色 proxy 是为了保证核心增强变量先稳定跑通。如果核心模型完成后还想加水色，再把 `USE_LANDSAT_WATER_COLOR = True`，同时保持 `FORCE_REEXTRACT_EE = True` 重新导出。
- 若社区资产 `HydroLAKES` 或 `GLOBathy` 在你的账号里不可用，把 `USE_HYDROLAKES` 或 `USE_GLOBATHY` 改成 `False`。